In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import dill

In [3]:
class BPE:
    def __init__(self, vocab_size: int):
        self.vocab_size = vocab_size
        self.id2token = None
        self.token2id = None
    
    def fit(self, text: str):
        uniq_tokens = sorted(set(text))
        tokens_list = list(text)

        while len(uniq_tokens) < self.vocab_size:
            print(len(uniq_tokens), '|', self.vocab_size)
            pairs = {}

            for i in range(len(tokens_list)-1):
                if (tokens_list[i], tokens_list[i+1]) not in pairs:
                    pairs[(tokens_list[i], tokens_list[i+1])] = 0
                pairs[(tokens_list[i], tokens_list[i+1])] += 1
            
            if len(pairs) == 0: break

            top_freq = 0
            top_pair = None
            for pair, freq in pairs.items():
                if freq > top_freq:
                    top_freq = freq
                    top_pair = pair
            
            uniq_tokens.append(top_pair[0]+top_pair[1])

            upd_tokens_list = []
            i=0
            while i < len(tokens_list):
                if (i < len(tokens_list) - 1 and tokens_list[i:i+2] == list(top_pair)):
                    upd_tokens_list.append(top_pair[0]+top_pair[1])
                    i+=2
                else:
                    upd_tokens_list.append(tokens_list[i])
                    i+=1

            tokens_list = upd_tokens_list
        
        self.id2token = {i: token for i, token in enumerate(uniq_tokens[:self.vocab_size])}
        self.token2id = {token: i for i, token in self.id2token.items()}

    def encode(self, text: str) -> list[int]:

        tokens_list = list(text)

        fin_flag = True
        while fin_flag:
            fin_flag = False

            upd_tokens_list = []
            i=0
            while i < len(tokens_list):
                if (i < len(tokens_list) - 1 and tokens_list[i]+tokens_list[i+1] in self.token2id):
                    upd_tokens_list.append(tokens_list[i]+tokens_list[i+1])
                    i+=2
                    fin_flag = True
                else:
                    upd_tokens_list.append(tokens_list[i])
                    i+=1
            
            tokens_list = upd_tokens_list
        
        token_id_list = []

        for token in tokens_list:
            token_id_list.append(self.token2id[token])
        
        return(token_id_list)
    
    def decode(self, token_ids: list[int]) -> str:

        return ''.join(self.id2token[i] for i in token_ids)

    def save(self, filename):
        with open(filename, 'wb') as f:
            dill.dump(self, f)
        print(f"Объект сохранён в {filename}")

    @classmethod
    def load(cls, filename):
        with open(filename, 'rb') as f:
            obj = dill.load(f)
                
        print(f"Объект загружен из {filename}")
        return obj


In [4]:
drova = "На дворе дрова, за двором дрова, дрова вширь двора, не вместит двор дров, надо дрова выдворить на дровяной двор."

In [5]:
bpe = BPE(28)
bpe.fit(drova)

print(bpe.id2token)
print(bpe.encode('вором дрова, дрова вширь двора, не вмест'))
print(bpe.decode(bpe.encode('вором дрова, дрова вширь двора, не вмест')))

21 | 28
22 | 28
23 | 28
24 | 28
25 | 28
26 | 28
27 | 28
{0: ' ', 1: ',', 2: '.', 3: 'Н', 4: 'а', 5: 'в', 6: 'д', 7: 'е', 8: 'з', 9: 'и', 10: 'й', 11: 'м', 12: 'н', 13: 'о', 14: 'р', 15: 'с', 16: 'т', 17: 'ш', 18: 'ы', 19: 'ь', 20: 'я', 21: ' д', 22: 'ро', 23: 'во', 24: ' дро', 25: ' дров', 26: ' дво', 27: ' двор'}
[23, 22, 11, 25, 4, 1, 25, 4, 0, 5, 17, 9, 14, 19, 27, 4, 1, 0, 12, 7, 0, 5, 11, 7, 15, 16]
вором дрова, дрова вширь двора, не вмест


In [6]:
class TokenEmbeddings(nn.Module):
    def __init__(self, vocab_size: int, emb_size: int):
        super().__init__()
        self.vocab_size = vocab_size
        self.emb_size = emb_size

        self.emb_matrix = nn.Embedding(vocab_size, emb_size)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        
        return self.emb_matrix(x)

In [7]:
class PositionalEmbeddings(nn.Module):
    def __init__(self, max_seq_len: int, emb_size: int):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.emb_size = emb_size

        self.emb_matrix = nn.Embedding(max_seq_len, emb_size)
    
    def forward(self, seq_len: torch.Tensor) -> torch.Tensor:

        return self.emb_matrix(torch.arange(0, seq_len))

In [8]:
class HeadAttention(nn.Module):
    def __init__(self, emb_size: int, head_size: int, max_seq_len: int):
        super().__init__()
        self.emb_size = emb_size
        self.head_size = head_size
        self.max_seq_len = max_seq_len
        self.w_k = nn.Linear(self.emb_size, self.head_size)
        self.w_q = nn.Linear(self.emb_size, self.head_size)
        self.w_v = nn.Linear(self.emb_size, self.head_size)
        self.mask_attention = torch.tril(torch.ones(max_seq_len, max_seq_len))
    
    def forward(self, x: float):
        seq_len = x.shape[1]

        self.attention_matrix = torch.matmul(self.w_q(x), self.w_k(x).transpose(1, 2))
        self.attention_matrix /= np.sqrt(self.head_size)
        self.mask_matrix = self.mask_attention[:seq_len, :seq_len]

        self.attention_matrix = torch.where(self.mask_matrix == 1,
                                            self.attention_matrix,
                                            torch.tensor(float('-inf'), device=self.attention_matrix.device, dtype=self.attention_matrix.dtype))
        
        self.attention_matrix = torch.softmax(self.attention_matrix, dim=2)

        return torch.matmul(self.attention_matrix, self.w_v(x))

In [9]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads: int, emb_size: int, head_size: int, max_seq_len: int, dropout: float = 0.1):
        super().__init__()
        self.num_heads = num_heads
        self.emb_size = emb_size
        self.head_size = head_size
        self.max_seq_len = max_seq_len

        self.heads = nn.ModuleList([HeadAttention(emb_size, head_size, max_seq_len)] * num_heads)
        self.linear = nn.Linear(self.head_size * self.num_heads, self.emb_size)
        self.drop = torch.nn.Dropout(p=dropout)
    
    def forward(self, x):

        outp_heads = []
        for head in self.heads:
            outp_heads.append(head(x))
        
        conc = torch.cat(outp_heads, dim=2)
        linear_res = self.linear(conc)
        
        return self.drop(linear_res)

In [10]:
class FeedForward(nn.Module):
    def __init__(self, emb_size: int, dropout: float = 0.1):
        super().__init__()
        self.emb_size = emb_size
        self.dropout = dropout

        self.lin1 = nn.Linear(emb_size, emb_size*4)
        self.relu1 = nn.ReLU()
        self.lin2 = nn.Linear(emb_size*4, emb_size)
        self.drop = torch.nn.Dropout(p=dropout)
    
    def forward(self, x):

        return(self.drop(self.lin2(self.relu1(self.lin1(x)))))

In [11]:
class Decoder(nn.Module):
    def __init__(self, num_heads:int, emb_size:int, head_size:int, max_seq_len: int, dropout: float=0.1):
        super().__init__()
        self.multi_head = MultiHeadAttention(num_heads, emb_size, head_size, max_seq_len, dropout)
        self.feed_forward = FeedForward(emb_size, dropout)

        self.norm1 = nn.LayerNorm(emb_size)
        self.norm2 = nn.LayerNorm(emb_size)
    
    def forward(self, x):
        x_proc = self.multi_head(x)
        x_proc += x
        x_proc = self.norm1(x_proc)
        x_proc = self.feed_forward(x_proc)
        x_proc += x
        x_proc = self.norm2(x)

        return x_proc

In [15]:
from turtle import forward


class GPT(nn.Module):
    def __init__(self, vocab_size: int, max_seq_len: int, emb_size: int, num_heads: int, head_size: int, num_layers: int, dropout: float=0.1, device: str='cpu'):
        super().__init__()

        self.token_emb = TokenEmbeddings(vocab_size, emb_size)
        self.pos_emb = PositionalEmbeddings(max_seq_len, emb_size)
        self.drop = nn.Dropout(dropout)

        self.decoders = nn.Sequential([Decoder(num_heads, emb_size, head_size, max_seq_len, dropout)] * num_layers)

        self.lin = nn.Linear(emb_size, vocab_size)

    def forward(self, x):
        x_embs = self.token_emb(x) + self.pos_emb(x.shape[1])
        x_dec = self.decoders(self.drop(x_embs))

        return self.lin(x_dec)
    
    def generate(self, x, max_new_tokens: int):
        for i in range(max_new_tokens):
            tokens = x[:, -self.max_seq_len:]
            logits = self.forward(tokens)[:, -1, :]
            sm_logits = torch.softmax(logits, dim=1)

            logits_upd = torch.reshape(torch.argmax(sm_logits, dim=1), (sm_logits.shape[0], 1))

            return torch.cat([x, logits_upd], dim=1)
    
    def save(self, path):
        torch.save({
            'model_state_dict': self.state_dict(),
            'vocab_size': self.vocab_size,
            'max_seq_len': self.max_seq_len,
            'emb_size': self.emb_size,
            'num_heads': self.num_heads,
            'head_size': self.head_size,
            'num_layers': self.num_layers
        }, path)
    
    @classmethod
    def load(cls, path, device):
        checkpoint = torch.load(path, map_location=device)
        model = cls(
            vocab_size=checkpoint['vocab_size'],
            max_seq_len=checkpoint['max_seq_len'],
            emb_size=checkpoint['emb_size'],
            num_heads=checkpoint['num_heads'],
            head_size=checkpoint['head_size'],
            num_layers=checkpoint['num_layers']
        )
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)
        return model